## Goal

Prediction multifamily prices

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

//anaconda/envs/uatu/lib/python3.7/site-packages/pandas/compat/_optional.py:138: UserWarning: Pandas requires version '2.7.0' or newer of 'numexpr' (version '2.6.9' currently installed).
  warnings.warn(msg, UserWarning)
//anaconda/envs/uatu/lib/python3.7/site-packages/sklearn/linear_model/least_angle.py:30: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  method='lar', copy_X=True, eps=np.finfo(np.float).eps,
//anaconda/envs/uatu/lib/python3.7/site-packages/sklearn/linear_model/least_angle.py:167: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and

## Load price data

In [2]:
fn = "../data/multifamily_prices.tsv"

df = pd.read_csv(fn, sep="\t")

## Review data

In [5]:
df.head(3)

,Address,size,lot size,bedrooms,bathroom,built year,Age,units,address,Ask Price,Link,Sold Price
0,25-27 Morrell,1888,NaN,4,2.0,1923.0,98.0,2.0,nobhill,1750000,https://www.redfin.com/CA/San-Francisco/25-Mor...,NaN
1,1133 - 1135 Judah St,2472,NaN,4,2.0,1946.0,75.0,2.0,sunset,1698000,https://www.redfin.com/CA/San-Francisco/1133-J...,NaN
2,1572-1576 Bush St,5113,1754.0,12,4.0,1907.0,114.0,NaN,eastbush,1750000,https://www.redfin.com/CA/San-Francisco/1572-B...,NaN


In [8]:
## Exctract all features

# skip index 0 which is Morrel
start_inx = 1

x_size = df['size'].tolist()[start_inx:]
assert len(x_size) == df.shape[0] - 1

x_bed = df['bedrooms'].tolist()[start_inx:]
assert len(x_bed) == df.shape[0] - 1

x_bath = df['bathroom'].tolist()[start_inx:]
assert len(x_bath) == df.shape[0] - 1

x_age = df['Age'].tolist()[start_inx:]
assert len(x_age) == df.shape[0] - 1

y_ = df['Ask Price'].tolist()[start_inx:]
assert len(y_) == df.shape[0] - 1

## Handle NANs

In [39]:
# Replace nan with mean
def impute(fet):
    impued_fet = []
    sum_, cnt = 0, 0
    nan_inxs = []
    for inx, x in enumerate(fet):
        if not np.isnan(x):
            impued_fet.append(x)
            sum_ += x
            cnt += 1
        else:
            nan_inxs.append(inx)
    avg = sum_/cnt
    for inx in nan_inxs:
        fet[inx] = avg
    
    return fet

## Impute all numerical feature

In [40]:
imputed_x_size = impute(x_size)
imputed_x_bed = impute(x_bed)
imputed_x_bath = impute(x_bath)
imputed_x_age = impute(x_age)
imputed_y_ = impute(y_)

In [44]:
print("Size of imputed_x_size: {}".format(len(imputed_x_size)))

print("Size of imputed_y_: {}".format(len(imputed_y_)))

Size of imputed_x_size: 11
Size of imputed_y_: 11


## Normalize numerical features

In [45]:
max_x_size = max(x_size)
x_size_norm = [1.0*x/max_x_size for x in x_size]

max_x_bed = max(x_bed)
x_bed_norm = [1.0*x/max_x_bed for x in x_bed]

max_x_bath = max(x_bath)
x_bath_norm = [1.0*x/max_x_bath for x in x_bath]

max_x_age = max(x_age)
x_age_norm = [1.0*x/max_x_age for x in x_age]

max_y_ = max(y_)
y_norm = [1.0*x/max_y_ for x in y_]

In [46]:
print("Size of y_norm: {}".format(len(y_norm)))

Size of y_norm: 11


In [47]:
y_norm

[0.5957894736842105,
 0.6140350877192983,
 0.6298245614035087,
 0.5877192982456141,
 0.7,
 0.5596491228070175,
 0.7017543859649122,
 1.0,
 0.736140350877193,
 0.631578947368421,
 0.6842105263157895]

## Construct X & y

In [48]:
X = np.array([x_size, x_bed, x_bath, x_age])
X_norm = np.array([x_size_norm, x_bed_norm, x_bath_norm, x_age_norm])

X = np.transpose(X)
X_norm = np.transpose(X_norm)

y = np.transpose(np.array([y_]))
y_norm = np.transpose(np.array([y_norm]))

In [50]:
X.shape

(11, 4)

In [51]:
y.shape

(11, 1)

## Train model

In [52]:
reg = LinearRegression().fit(X, y)

reg_2 = LinearRegression().fit(X_norm, y_norm)

## Review model

In [53]:
reg.coef_

array([[   119.90375568, -62765.34326066,  65902.25554793,
           148.23401827]])

In [54]:
reg.intercept_

array([1749879.17889384])

In [55]:
reg_2.coef_

array([[ 0.23816672, -0.26427513,  0.11561799,  0.00629344]])

In [56]:
reg_2.intercept_

array([0.61399269])

## Predict  Morrel

In [57]:
x_morrel = np.array([[1888, 4, 2.0, 98]])
x_morrel_norm = np.array([[1888.0/max_x_size, 4.0/max_x_bed, 2.0/max_x_bath, 98.0/max_x_age]])

y_morrel = reg.predict(x_morrel)
y_morrel_2 = reg_2.predict(x_morrel_norm)

In [58]:
y_morrel

array([[1871527.54146896]])

In [59]:
y_morrel_2*max_y_

array([[1871527.54146896]])

In [60]:
ask_price = 1750000
prediction = 1871527.54146896

In [61]:
(prediction-ask_price)/prediction

0.06493494686889467

In [62]:
offer_0 = 1800000
offer_1 = 1700000
offer_2 = 1690000
offer_3 = 1680000
offer_4 = 1675000
offer_5 = 1670000
offer_6 = 1650000

In [63]:
(ask_price-offer_0)/ask_price

-0.02857142857142857

In [64]:
(ask_price-offer_1)/ask_price

0.02857142857142857

In [104]:
(ask_price-offer_2)/ask_price

0.03428571428571429

In [105]:
(ask_price-offer_3)/ask_price

0.04

In [106]:
(ask_price-offer_4)/ask_price

0.04285714285714286

In [107]:
(ask_price-offer_5)/ask_price

0.045714285714285714

In [108]:
(ask_price-offer_6)/ask_price

0.05714285714285714

In [112]:
1750000-72000

1678000

In [113]:
(ask_price-1670000)/ask_price

0.045714285714285714